In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
ls

datasets/   main.ipynb  README.md


In [3]:
df = pd.read_csv('datasets/StealthPhisher2025.csv')

In [4]:
df.head(10)

,URL,LengthOfURL,Domain,URLComplexity,CharacterComplexity,DomainLengthOfURL,IsDomainIP,TLD,TLDLength,LetterCntInURL,...,UniqueFeatureCnt,WAPLegitimate,WAPPhishing,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt,LikelinessIndex,Label
0,https://bafkreibre4pwizu3d73y7at37ewy6nhklfhb4...,84,bafkreibre4pwizu3d73y7at37ewy6nhklfhb4mb75tp25...,14.000,0.013158,76,0,com,3,55,...,0,0.036909,0.034343,4.739567,1.00,1.000000,1,3,0.263200,Phishing
1,http://101.200.220.118:8090/ledshow2.exe,40,101.200.220.118:8090,12.000,0.030303,20,1,118:8090,8,10,...,0,0.030220,0.031712,3.808271,1.00,1.222222,5,3,0.329167,Phishing
2,https://1llc5nv.duckdns.org/,28,1llc5nv.duckdns.org,9.000,0.050000,19,0,org,3,15,...,0,0.043893,0.038140,4.056021,0.75,1.307692,0,2,0.684211,Phishing
3,http://hrga.melonwoodhomes.com/,31,hrga.melonwoodhomes.com,9.375,0.041667,23,0,com,3,21,...,0,0.061778,0.046342,3.827833,0.75,1.275862,0,2,0.608696,Phishing
4,https://www.aspb.gob.bo,23,www.aspb.gob.bo,5.875,0.066667,15,0,bo,2,9,...,10,0.073055,0.046233,3.346439,1.00,1.400000,0,2,0.733333,Legitimate
5,https://s.kurersekoyasszua-jp.icu/,34,s.kurersekoyasszua-jp.icu,10.250,0.038462,25,0,icu,3,22,...,0,0.049507,0.042152,3.965018,0.75,1.250000,0,3,0.440000,Phishing
6,https://www.ekli-nat.ekinrt.lnedxt.top,38,www.ekli-nat.ekinrt.lnedxt.top,8.750,0.033333,30,0,top,3,22,...,0,0.065719,0.047155,3.917626,1.00,1.235294,0,3,0.433333,Phishing
7,https://www.avesnocturnas.es,28,www.avesnocturnas.es,6.000,0.050000,20,0,es,2,15,...,10,0.074066,0.048185,3.719295,1.00,1.307692,0,2,0.550000,Legitimate
8,https://qrco.de/bfLW8S,22,qrco.de,7.750,0.071429,7,0,de,2,11,...,0,0.050429,0.043513,4.070656,1.00,1.380952,1,2,1.000000,Phishing
9,https://www.salsapower.com,26,www.salsapower.com,5.750,0.055556,18,0,com,3,13,...,1,0.079889,0.048268,3.636842,1.00,1.333333,0,2,0.666667,Legitimate


In [5]:
# Hedef: Label (Phishing / Legitimate)
y = df["Label"].map({"Legitimate": 0, "Phishing": 1})
X = df.drop(columns=["Label"])

# Stratified split (sınıf oranını korur)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(df.shape, y.value_counts(normalize=True))

(336749, 65) Label
1    0.522068
0    0.477932
Name: proportion, dtype: float64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# 1) Kullanılacak sütunlar
categorical_cols = [c for c in ["TLD", "IsDomainIP"] if c in X_train.columns]
drop_text_cols = [c for c in ["URL", "Domain"] if c in X_train.columns]

# Sayısal sütunlar: geriye kalanlardan kategorik ve drop edilecekleri çıkar
numeric_cols = [c for c in X_train.columns if c not in categorical_cols + drop_text_cols]

# 2) Preprocess
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))  # sparse uyumlu
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_A = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop"
)

def eval_model(name, model, X_test, y_test):
    proba = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)

    print(f"\n===== {name} =====")
    print(classification_report(y_test, preds, digits=4))
    if proba is not None:
        print("ROC-AUC:", roc_auc_score(y_test, proba))

# 3) Algoritma 1: Logistic Regression
pipe_lr = Pipeline(steps=[
    ("prep", preprocess_A),
    ("clf", LogisticRegression(max_iter=200, n_jobs=-1, class_weight="balanced"))
])

# 4) Algoritma 2: Random Forest
pipe_rf = Pipeline(steps=[
    ("prep", preprocess_A),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        class_weight="balanced",
        random_state=42
    ))
])

# 5) Algoritma 3: HistGradientBoosting (genelde çok iyi baseline verir)
pipe_hgb = Pipeline(steps=[
    ("prep", preprocess_A),
    ("clf", HistGradientBoostingClassifier(
        max_depth=8,
        learning_rate=0.08,
        max_iter=300,
        random_state=42
    ))
])

# Fit + Evaluate
pipe_lr.fit(X_train, y_train)
eval_model("A1 - LogisticRegression (Tablolu)", pipe_lr, X_test, y_test)

pipe_rf.fit(X_train, y_train)
eval_model("A2 - RandomForest (Tablolu)", pipe_rf, X_test, y_test)

pipe_hgb.fit(X_train, y_train)
eval_model("A3 - HistGradientBoosting (Tablolu)", pipe_hgb, X_test, y_test)



===== A1 - LogisticRegression (Tablolu) =====
              precision    recall  f1-score   support

           0     0.9954    0.9994    0.9974     32189
           1     0.9995    0.9958    0.9976     35161

    accuracy                         0.9975     67350
   macro avg     0.9974    0.9976    0.9975     67350
weighted avg     0.9975    0.9975    0.9975     67350

ROC-AUC: 0.9999182336011473

===== A2 - RandomForest (Tablolu) =====
              precision    recall  f1-score   support

           0     0.9984    0.9993    0.9989     32189
           1     0.9993    0.9986    0.9989     35161

    accuracy                         0.9989     67350
   macro avg     0.9989    0.9989    0.9989     67350
weighted avg     0.9989    0.9989    0.9989     67350

ROC-AUC: 0.9999478228183889


TypeError: Sparse data was passed for X, but dense data is required. Use '.toarray()' to convert to a dense numpy array.